In [1]:
import pandas as pd
import os
import numpy as np

# Focus on Sectors with Granular Data

The ASPBI dataset contains employment data for various sectors. Fourteen of these sectors are provided in a semi-standardized format, which allows for both aggregated and subsector-level analysis. However, there are still inconsistencies with the formatting, so it’s necessary to keep two DataFrames. One contains aggregated job totals, which are more reliable, while the other holds granular subsector data, which is less trustworthy due to formatting issues. This way, analysis can rely on the aggregated numbers while still preserving the detailed data for reference. The remaining four sectors do not have any granular data available, so only total figures can be used for analysis.

### Sectors with semi-standardized data (14)
- Accommodation and Food Service Activities
- Administrative and Support Service Activities
- Agriculture, Forestry, and Fishing
- Arts, Entertainment and Recreation
- Construction
- Electricity, Gas, Steam and Air Conditioning Supply
- Education
- Financial and Insurance Activities
- Human Health and Social Work Activities
- Information and Communication
- Manufacturing
- Other Service Activities
- Real Estate Activities
- Transportation and Storage

### Sectors with no granular data (4)
- Mining and Quarrying
- Professional, Scientific and Technical Activities
- Public Administration and Defense; Compulsory Social Security
- Wholesale and Retail Trade; Repair of Motor Vehicles and Motorcycles




In [2]:
# Merge the different excel files the contain data per job subsector (called industry by PSA)
primary_file = 'industry'

aggregated_dfs = []
granular_dfs = []

column_names = [
    '2009 PSIC Code',
    'Region Description',
    'Job Subsector',
    'Total Male',
    'Total Female',
    'Paid Male',
    'Paid Female',
    'Unpaid Male',
    'Unpaid Female'
]

cols_human_resources = ['Total Male',
    'Total Female',
    'Paid Male',
    'Paid Female',
    'Unpaid Male',
    'Unpaid Female'
]

section_mapping = {
    'A': 'Agriculture, Forestry, and Fishing',
    'B': 'Mining and Quarrying',
    'C': 'Manufacturing',
    'D': 'Electricity, Gas, Steam and Air Conditioning Supply',
    'E': 'Water Supply; Sewerage, Waste Management and Remediation',
    'F': 'Construction',
    'G': 'Wholesale and Retail Trade; Repair of Motor Vehicles and Motorcycles',
    'H': 'Transportation and Storage',
    'I': 'Accommodation and Food Service Activities',
    'J': 'Information and Communication',
    'K': 'Financial and Insurance Activities',
    'L': 'Real Estate Activities',
    'M': 'Professional, Scientific and Technical Activities',
    'N': 'Administrative and Support Service Activities Sector',
    'O': 'Public Administration and Defense; Compulsory Social Security',
    'P': 'Education',
    'Q': 'Human Health and Social Work Activities',
    'R': 'Arts, Entertainment and Recreation',
    'S': 'Other Service Activities'
}

In [3]:
# For each file in the 'industry' file
primary_file = 'industry'
for excel_file in os.listdir(primary_file):
    # Open the file
    filepath = os.path.join(primary_file, excel_file)
    print(filepath)

    # Get the section letter
    section_letter = excel_file[8]
    sector_name = section_mapping[section_letter]

    # Read the first 20 rows just to find the header
    preview = pd.read_excel(filepath, sheet_name=2, header=None, nrows=20)

    # Find the row that contains '2009 PSIC' and start the df there
    header_row = preview[preview.apply(lambda row: row.astype(str).str.contains("2009").any(), axis=1)].index[0]
    df = pd.read_excel(filepath, sheet_name=2, header=header_row, skipfooter=5).iloc[3:]

    # Get the first 8 or 9 columns
    df = df.iloc[:, :9]  

    # 9 columns, rename them
    if len(df.columns) == 9:
        df.columns = column_names
    # if 8 columns, then add a new column and reorganized
    if len(df.columns) == 8:
        df.columns = [name for name in column_names if name != 'Job Subsector']
        df['Job Subsector'] = df['Region Description'].copy()
        df = df[column_names]

    # Get the rows that are for the region
    is_region_row = df['2009 PSIC Code'].isnull()

    # CLEAN Job Subsector
    df['Job Subsector'] = (
        df['Job Subsector']
        .astype(str)           # ensure strings
        .str.replace('\n',' ', regex=True)  # remove newlines
        .str.strip()           # remove leading/trailing spaces
        .str.title()           # capitalize each word
    )

    # If the row is for the industry, then it is not for the region and vice-versa
    df['Job Subsector'] = df.loc[~is_region_row, 'Region Description'].copy()
    df['Region Description'] = df.loc[is_region_row,'Region Description']
    df['Region Description'].ffill(inplace=True)

    # Add additional information like the section letter and job sector
    section_letter = excel_file[8]
    df['Section Letter'] = section_letter
    df['Job Sector'] = sector_name
    # if '-' means 0 and if 's', then suppresed for confidentiality, so np.nan
    df.replace({'-':0, 's':np.nan}, inplace=True)

    # Lets finalize the aggragated dataframe
    aggregated_dfs.append(df[is_region_row])

    # You need to also filter out the sums 
    # from the aggregate cuz you dont want to overcount
    df[cols_human_resources] = df.loc[~is_region_row, cols_human_resources]
    granular_dfs.append(df)

granular_df = pd.concat(granular_dfs)


aggregated_df = pd.concat(aggregated_dfs)
# Remove rows where 'Region Description' is the string 'nan'
aggregated_df = aggregated_df[aggregated_df['Region Description'].str.lower() != 'nan']
# Just clean the names to the official versions
region_mapping = {
    'PHILIPPINES': 'Philippines',
    'National Capital Region': 'National Capital Region (NCR)',
    'Cordillera Administrative Region': 'Cordillera Administrative Region (CAR)',
    'I - Ilocos Region': 'Region I (Ilocos Region)',
    'II - Cagayan Valley': 'Region II (Cagayan Valley)',
    'III - Central Luzon': 'Region III (Central Luzon)',
    'IV-A - CALABARZON': 'Region IV-A (CALABARZON)',
    'MIMAROPA Region': 'MIMAROPA Region',
    'V - Bicol Region': 'Region V (Bicol Region)',
    'VI - Western Visayas': 'Region VI (Western Visayas)',
    'VII - Central Visayas': 'Region VII (Central Visayas)',
    'VIII - Eastern Visayas': 'Region VIII (Eastern Visayas)',
    'IX - Zamboanga Peninsula': 'Region IX (Zamboanga Peninsula)',
    'X - Northern Mindanao': 'Region X (Northern Mindanao)',
    'XI - Davao Region': 'Region XI (Davao Region)',
    'XII - SOCCSKSARGEN': 'Region XII (SOCCSKSARGEN)',
    'XIII - Caraga': 'Region XIII (Caraga)',
    'Bangsamoro Autonomous Region in Muslim Mindanao': 'Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)',
    'BARMM': 'Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)',
    'Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)': 'Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)',
    'Bangsamoro Autonomous Region \nin Muslim Mindanao (BARMM)': 'Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)'
}
aggregated_df['Region Description'] = aggregated_df['Region Description'].apply(lambda x: region_mapping[x])
granular_df['Region Description'] = granular_df['Region Description'].apply(lambda x: region_mapping[x])


industry/Section D - Statistical Tables by Region and Industry Group_2022 ASPBI Final Results.xlsx
industry/Section S - Statistical Tables by Region and Industry Group_2022 ASPBI Final Results.xlsx
industry/Section H - Statistical Tables by Region and Industry Group_2022 ASPBI Final Results.xlsx
industry/Section G - Statistical Tables by Region and Industry Group_2022 ASPBI Final Results.xlsx
industry/Section P - Statistical Tables by Region and Industry Group_2022 ASPBI Final Results.xlsx
industry/Section N - Statistical Tables by Region and Industry Group_2022 ASPBI Final Results_05062025_0.xlsx
industry/Section K - Statistical Tables by Region and Industry Group_2022 ASPBI Final Results.xlsx
industry/Section J - Statistical Tables by Region and Industry Group_2022 ASPBI Final Results_05052025.xlsx
industry/Section L -Statistical Tables by Region and Industry Group_2022 ASPBI Final Results.xlsx
industry/Section Q - Statistical Tables by Region and Industry Group_2022 ASPBI Final Resu

In [4]:
aggregated_df['Total'] = aggregated_df['Total Male'] + aggregated_df['Total Female']
aggregated_df['Paid'] = aggregated_df['Paid Male'] + aggregated_df['Paid Female']
relevant_cols = ['Region Description', 'Job Sector', 'Total', 'Paid']
aggregated_relevant_df = aggregated_df[relevant_cols]

# Focus on Sectors with No Granular Data

Manually add in the data from the sectors with no granular data

In [5]:
preliminary_files = [
    '2-F_2022-ASPBI-Summary-Statistics-and-Selected-Indicators_LBC_082024.xlsx', 
    '2. A_2022-ASPBI-Summary-Statistics-and-Selected-Indicatorsv5.xlsx',
    '2. B_2022 ASPBI_Summary Statistics and Selected Indicators_v1.xlsx',
    '2-C_2022ASPBI_SSSI.xlsx'
]

In [7]:
construction_df

,Unnamed: 1,Job Sector,Unnamed: 3,Unnamed: 4
0,Philippines,Construction,286849.0,286251.0
2,National Capital Region (NCR) ...,Construction,160781.0,160748.0
3,Cordillera Administrative Region (CAR) ...,Construction,2398.0,2384.0
4,Region I (Ilocos Region) ...,Construction,2454.0,2427.0
5,Region II (Cagayan Valley) ...,Construction,1982.0,1966.0
6,Region III (Central Luzon) ...,Construction,11876.0,11869.0
7,Region IV-A (CALABARZON) ...,Construction,34964.0,34937.0
8,MIMAROPA Region ...,Construction,1746.0,1728.0
9,Region V (Bicol Region) ...,Construction,6911.0,6905.0
10,Region VI (Western Visayas) ...,Construction,9731.0,9714.0


In [8]:
construction_df = pd.read_excel('preliminary/' + preliminary_files[0], 
                                sheet_name=2, 
                                usecols=[1, 3, 4], 
                                skiprows=7, skipfooter=31).drop(index=1)
construction_df.insert(1, 'Job Sector', 'Construction')

aggregated_cols = ['Regional Description', 'Job Sector', 'Total', 'Paid']
construction_df.columns = aggregated_cols

In [9]:
agri_df = (
    pd.read_excel(
        'preliminary/' + preliminary_files[1],
        sheet_name=2,
        usecols=[0, 2, 3],
        skiprows=8,
        skipfooter=31
    )
    .drop(index=1)
    .replace({
        'Bangsamoro Autonomous Region \nin Muslim Mindanao (BARMM)     ':
        'Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)'
    })
)

# Insert Job Sector as the second column
agri_df.insert(1, 'Job Sector', 'Agriculture, Forestry, and Fishing')

# Set final column order
agri_df.columns = aggregated_cols

In [10]:
# Read Excel and drop second row
mining_df = pd.read_excel(
    'preliminary/' + preliminary_files[2],
    sheet_name=2,
    usecols=[1, 3, 4],
    skiprows=7,
    skipfooter=26
).drop(index=1)

# Insert new column 'Job Sector' at position 1 (second column)
mining_df.insert(1, 'Job Sector', 'Mining and Quarrying')

# Rename other columns if needed
mining_df.columns = ['Region Description', 'Job Sector', 'Total', 'Paid']

In [11]:
manufacturing_df = pd.read_excel('preliminary/' + preliminary_files[3], 
                        sheet_name=3,
                        usecols=[0, 2, 3],
                        skiprows=7).drop(index=1)

# Insert new column 'Job Sector' at position 1 (second column)
manufacturing_df.insert(1, 'Job Sector', 'Manufacturing')
manufacturing_df.columns = aggregated_cols

In [12]:
final_aggregated_df = pd.concat([aggregated_relevant_df, construction_df, agri_df, mining_df, manufacturing_df])
final_aggregated_df['Region Description'] = final_aggregated_df['Region Description'].str.strip()
final_aggregated_df.replace({'s':np.nan, '-':0}, inplace=True)
final_aggregated_df.to_csv('aggregated_aspbi.csv', index=False)

In [13]:
granular_df.to_csv('granular_aspbi.csv', index=False)